# Tutorial 2: hop-bound-preserving sparsification in depth

This notebook explores the difference between two sparsification modes:

1. `reachq.research.sparsify.sparsify_shortcut_set` (reachability only):
   removes shortcuts whose removal preserves reachability but
   may violate the hopbound.
2. `reachq.research.sparsify_hop.sparsify_hop_bounded` (hop-bound-preserving):
   removes shortcuts whose removal preserves the hopbound.

We construct a graph where the JLS's hopbound is tighter than the
graph's natural diameter. This forces the two modes to diverge.

Graph: a 'chain of stars' where each star's leaves can only reach
the next star via a single bridge edge. The hopbound is the
minimum such that the bridge edges are within reachability.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from reachq.core.algorithm import build_shortcut_set_for_reachability
from reachq.graph import Digraph
from reachq.reachability import bfs_reachability, parallel_bfs
from reachq.research.sparsify import sparsify_shortcut_set
from reachq.research.sparsify_hop import sparsify_hop_bounded

In [ ]:
# Build a 'chain of stars' graph
def chain_of_stars(k, n):
    g = Digraph()
    for i in range(k):
        center = ("c", i)
        g.add_vertex(center)
        for j in range(n):
            leaf = ("l", i, j)
            g.add_vertex(leaf)
            g.add_edge(center, leaf)
            g.add_edge(leaf, center)
        if i > 0:
            g.add_edge(("c", i - 1), center)
    return g


g = chain_of_stars(k=5, n=4)
print(f"Graph: {g.num_vertices()} vertices, {g.num_edges()} edges")

In [ ]:
# Construct the JLS shortcut set
H_jls, beta = build_shortcut_set_for_reachability(
    g, omega=3.0, random_seed=42, sparsify_shortcuts=False
)
print(f"JLS: |H|={len(H_jls)}, beta={beta:.2f}")

In [ ]:
# Compute empirical hop distance from vertex 0 with each mode
from collections import deque


def hop_max(g, source, H, beta):
    visited = {source: 0}
    q = deque([source])
    out = g.out_edges
    while q:
        u = q.popleft()
        if visited[u] >= beta:
            continue
        for v in out.get(u, ()):
            if v not in visited:
                visited[v] = visited[u] + 1
                q.append(v)
    return max(visited.values())


src = ("c", 0)
print(
    f"JLS (no sparsify):       |H|={len(H_jls)}, max-hop = {hop_max(g, src, H_jls, 999)}"
)
H_reach = sparsify_shortcut_set(g, H_jls)
print(
    f"reach-only sparsify:    |H|={len(H_reach)}, max-hop = {hop_max(g, src, H_reach, 999)}"
)
H_hop = sparsify_hop_bounded(g, H_jls, beta=int(beta) + 1)
print(
    f"hop-bound sparsify:     |H|={len(H_hop)}, max-hop = {hop_max(g, src, H_hop, int(beta) + 1)}, beta={beta:.2f}"
)

In [ ]:
# Verify reachability preservation for each mode
print("Soundness (reach-only):")
for s in g.vertices():
    if bfs_reachability(g, s) != parallel_bfs(g, s, H_reach):
        print(f"  Soundness VIOLATED at {s}")
        break
else:
    print("  preserved for all sources")

print("Soundness (hop-bound):")
for s in g.vertices():
    if bfs_reachability(g, s) != parallel_bfs(g, s, H_hop):
        print(f"  Soundness VIOLATED at {s}")
        break
else:
    print("  preserved for all sources")

In [ ]:
# Plot the size of |H| as a function of beta for each mode
import matplotlib.pyplot as plt


def size_at_beta(g, H, beta):
    from reachq.research.sparsify_hop import sparsify_hop_bounded

    return len(sparsify_hop_bounded(g, H, beta=beta))


betas = [5, 10, 15, 20, 30, 50]
hop_sizes = [size_at_beta(g, H_jls, b) for b in betas]

plt.figure(figsize=(8, 4))
plt.plot(betas, hop_sizes, "o-", label="hop-bound-preserving")
plt.axhline(
    y=len(H_reach), color="r", linestyle="--", label=f"reach-only (|H|={len(H_reach)})"
)
plt.xlabel("beta")
plt.ylabel("|H| after sparsification")
plt.title("Sparsification: |H| vs beta (chain of stars, k=5, n=4)")
plt.legend()
plt.grid(True)
plt.show()

## What you should see

1. The chain of stars has diameter 2k (where k is the number of
   stars). The JLS hopbound is computed from the graph's
   structural properties.
2. The JLS shortcut set has many redundant shortcuts (the
   graph is already well-connected within each star).
3. The reach-only sparsifier removes shortcuts but may
   increase the empirical hop-distance beyond beta.
4. The hop-bound-preserving sparsifier keeps shortcuts that are
   essential for maintaining the hopbound. As beta increases,
   the number of essential shortcuts decreases.

## What's next

Tutorial 3 explores iterative refinement: H_2 = JLS(G+H_1) versus H_1
= JLS(G). The shortcuts in H_1 that are not in H_2 are
"self-redundant" — JLS added them but wouldn't re-add them given
H_1 already in the graph.